# Week12 Attribution analysis — retrospective sensitivity, not internal reasoning

VALIDATION ONLY. No confirmation/TEST access. Default Run All uses clearly labelled synthetic fixtures. Real cached setup is allowed in J; real perturbations require explicit RUN_FULL in K. L reads saved results. Evidence attribution means context-dependent input ablation, not internal-model explanation. Optional attention/replacement/third-system studies remain disabled.

Order: H saved analysis → J (`RUN_SMOKE=False`, cached preparation) → K (`RUN_FULL=True`, real perturbations) → L (`RUN_SMOKE=False`, saved real analysis).

In [ ]:
RUN_SMOKE = True
RUN_FULL = False
RESUME = True
OVERWRITE = False
from pathlib import Path
import sys, subprocess, json
import pandas as pd
from IPython.display import display, Markdown, Image
ROOT=Path.cwd().resolve()
while not (ROOT/'src/week12').is_dir():
    if ROOT==ROOT.parent: raise FileNotFoundError('Open within project')
    ROOT=ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from src.week12.protocol import load, guard
from src.week12_attribution.protocol import directory
cfg=load();guard()
if OVERWRITE or not RESUME: raise ValueError('RESUME=True, OVERWRITE=False required')
SCRIPT=ROOT/'scripts/week12/run_attribution.py'
def run(stage, *args):
    subprocess.run([str(ROOT/'sara_env/bin/python'),str(SCRIPT),'--stage',stage,*args],cwd=ROOT,check=True)


In [ ]:
if RUN_FULL: raise ValueError('L is saved-output analysis only; generation belongs in K')
run('analyze',*(['--smoke'] if RUN_SMOKE else []))
f=directory(cfg,RUN_SMOKE)/'analysis'
for name in ['system_quality','unit_summary','relevance_utility','curve_summary','paired_harmful_effects','gold_overlap','win_loss_pairs']:
    p=f/(name+'.csv')
    if p.exists(): display(pd.read_csv(p))
if (f/'REPORT.md').exists(): display(Markdown((f/'REPORT.md').read_text()))
for p in sorted((f/'figures').glob('*.png')):display(Image(filename=str(p)))
if (f/'qualitative_candidates.json').exists():
    display(Markdown('## Candidate case studies — manual review required'))
    display(pd.DataFrame(json.loads((f/'qualitative_candidates.json').read_text())))


In [ ]:
# WIN/LOSS profiles remain retrospective; unit granularity and token length are explicit.
profile=f/'query_profiles.csv';units=f/'unit_importance.csv'
if profile.exists() and units.exists():
    q=pd.read_csv(profile);u=pd.read_csv(units)
    q['budget_utilization']=q.evidence_tokens/q.budget
    display(q.groupby(['dataset','budget','method','qbgf_outcome'])[['units','evidence_tokens','budget_utilization','harmful_fraction','importance_concentration','relevance_concentration','redundancy','gold_coverage']].agg(['mean','count']))
    g=u.groupby(['dataset','budget','method','granularity']).agg(n_units=('unit_id','size'),mean_unit_tokens=('tokens','mean'),mean_importance=('importance','mean'))
    display(g)
    print('Zero leave-one-out metric effect does not establish that evidence was ignored.')
    print('Top-m curves omit unavailable m; inspect curve_summary n before comparing curves.')
